# AgriStress-500 Dense Patch Token (DPT) Pipeline

Download the dataset -> sample images from one cluster -> tile each image and embed every tile via the PAI embedding API -> run Dense Patch Token evaluation, using the `precisionai.agriemb` package that wires the three sibling projects together:

- [`agristress-utils`](https://github.com/Precision-AI-Inc/agristress-utils) - dataset download and sampling
- [Precision AI Embeddings](https://embeddings.precision.ai/) - hosted image embedding API
- [`agrieval`](https://github.com/Precision-AI-Inc/agrieval) - Dense Patch Token evaluation

DPT scores the per-patch feature-map output of a dense vision backbone — the spatial grid of patch embeddings a model produces before pooling into a single image embedding. That's a different kind of artifact from the pooled, whole-image vectors the hosted PAI embedding API returns directly, so **no dense-feature backbone is wired up to that API** — instead, each sampled image is sliced into a `GRID_HEIGHT` x `GRID_WIDTH` grid of tiles, and every tile is embedded individually through the real API. Every value in the resulting feature map is a genuine embedding — only the tiling scheme is fabricated, not the features. Each tile is a separate `embeddings.create` call, so total API calls scale as `NUM_SAMPLES * GRID_HEIGHT * GRID_WIDTH` — keep the grid modest.

Validating a ground-truth mask for label-aware metrics requires decoding it at full AgriStress-500 resolution (several megapixels) before downsampling to the patch grid, which costs on the order of **a minute per image** — kept small with `NUM_SAMPLES = 2` below. Set `WITH_LABELS = False` to skip masks entirely for a near-instant, unsupervised-metrics-only result.

**Setup** (from the repo root, before starting Jupyter):

```bash
pip install -e ".[notebook]"
cp .env.example .env   # fill in PAI_EMBEDDINGS_BASE_URL and PAI_EMBEDDINGS_API_KEY
```

Run all cells top-to-bottom. See the root `README.md` and the standalone `examples/01_download_dataset.py` plus `examples/dpt/01_sample_cluster.py`-`03_run_eval.py` (or the bundled `04_end_to_end.py`) scripts for the non-notebook version.

In [ ]:
from pathlib import Path

from precisionai.agriemb import (
    build_client,
    download,
    embed_patch_tokens,
    run_dpt_evaluation,
    sample_cluster,
)
from precisionai.agriemb.config import DEFAULT_DATASET_DIR, DEFAULT_MODEL, DEFAULT_REPO_ID
from precisionai.agrieval.dpt import print_result
from precisionai.agristress.utils import AgriStressDataset

SEED = 0
NUM_SAMPLES = 4
GRID_HEIGHT = 12
GRID_WIDTH = 12
WITH_LABELS = True

## 1. Download the dataset

In [ ]:
dataset_dir = download(DEFAULT_DATASET_DIR, repo_id=DEFAULT_REPO_ID)
dataset = AgriStressDataset(dataset_dir)
clusters = dataset.list_clusters()

print(f"Downloaded to {dataset_dir}")
print(f"Clusters : {len(clusters)}")
print(f"Images   : {len(dataset.list_image_ids())}")

## 2. Sample a cluster

Picks a random L2 cluster and draws `NUM_SAMPLES` images from it. Pass `cluster="B1"` (or any id from `clusters` above) to target a specific one.

In [ ]:
cluster, sample = sample_cluster(dataset, cluster="B1", num_samples=NUM_SAMPLES, seed=SEED)
paths = dataset.resolve_paths(sample)
stems = {Path(image_id).stem: path for image_id, path in paths.items()}

print(f"Cluster {cluster}: sampled {len(sample)} of {len(dataset.list_image_ids(cluster=cluster))} images")
print("\n".join(sample))

## 3. Tile each image and embed every tile

`embed_patch_tokens` slices each image into a `[GRID_HEIGHT, GRID_WIDTH]` grid via `tile_image`, embeds every tile individually through the PAI embedding API, and stacks the real per-tile vectors into a `[P, H, W]` feature map per image (`P` is the embedding dimension the API returns).

In [ ]:
client = build_client()
images = embed_patch_tokens(client, stems, grid_size=(GRID_HEIGHT, GRID_WIDTH), model=DEFAULT_MODEL)

n_tiles = len(stems) * GRID_HEIGHT * GRID_WIDTH
print(f"Embedded {n_tiles} tiles into [H={GRID_HEIGHT}, W={GRID_WIDTH}] feature maps")

## 4. Run Dense Patch Token evaluation

`run_dpt_evaluation` pairs the feature maps with the dataset's ground-truth segmentation masks (`masks/`) and class map (`class_map.json`) for label-aware metrics, then evaluates them with `agrieval`'s `run_dpt_image_eval`. With `WITH_LABELS = True` this validates each image's full-resolution mask (~1 minute per image); set it to `False` above for a near-instant, unsupervised-metrics-only result.

In [ ]:
result = run_dpt_evaluation(dataset_dir, images, k_values=[5, 10], with_labels=WITH_LABELS)
print_result(result)